# 🎨 SIRCCD - Entrenamiento v4: YOLO11l Pre-entrenado (COCO)

## Estrategia
- **Modelo**: YOLO11l pre-entrenado en COCO (transfer learning)
- **Clases**: 2 — `bache`, `grieta` (sin cambios)
- **GPU objetivo**: H100 (80GB)
- **Resolución**: 1280 para máxima precisión en grietas finas
- **Baseline a superar**: v1 (YOLOv8m) — mAP50=0.734, P=0.733, R=0.683

## Modelos Disponibles

| Modelo | Params | GFLOPs | mAP50-95 COCO | VRAM |
|--------|--------|--------|---------------|------|
| YOLOv8m (v1) | 25.9M | 79.1 | 50.2 | ~8 GB |
| **YOLO11l** ⭐ | **25.3M** | **86.9** | **53.4** | **~10 GB** |
| YOLO11x | 56.9M | 195 | 54.7 | ~16 GB |
| YOLO26l | 24.8M | 86.4 | 55.0 | ~10 GB |
| YOLO26x | 55.7M | 193.9 | 57.5 | ~16 GB |

## ¿Por qué YOLO11l?
- **+3.2 mAP50-95** vs YOLOv8m en COCO (53.4 vs 50.2)
- **Parámetros similares** al v1 (25.3M vs 25.9M) → misma velocidad de inferencia
- **Arquitectura probada** y estable (> 1 año de madurez)
- Transfer learning desde COCO: converge rápido, aprovecha features pre-aprendidos

## Optimizaciones incluidas (de v3)

| Parámetro | v1 (baseline) | **v4 (optimizado)** | Mejora |
|-----------|---------------|---------------------|--------|
| **LRF** | 0.01 | **0.15** | Evita plateau |
| **IoU train** | 0.6 | **0.7** | Bboxes más precisos |
| **Label smoothing** | 0.05 | **0.0** | Mejor bbox regression |
| **Mixup** | 0.05 | **0.10** | Más diversidad |
| **Copy-paste** | 0.10 | **0.15** | Más variaciones |
| **Close mosaic** | 20 | **15** | Más epochs de refinamiento |
| **Conf (val)** | Default | **0.001** | mAP más preciso |
| **Conf (prod)** | 0.25 | **0.30** | Menos falsos positivos |
| **Val period** | 1 | **5** | Monitoreo frecuente |
| **Deduplicación** | Sí | **❌ NO** | Más datos, mejor generalización |

## Tiempo Estimado

| GPU | 100 epochs | 250 epochs | Costo Colab Pro+ |
|-----|------------|------------|------------------|
| **H100 80GB** | **~10-13 horas** | **~22-28 horas** | ~$36-60 / ~$75-120 |
| A100 80GB | ~20-25 horas | ~45-55 horas | ~$12-22 / ~$25-45 |

## Recomendación
1. Ejecutar con `EPOCHS = 100` para validar configuración
2. Si mejora vs v1 → continuar hasta 250 epochs con pause/resume
3. Evaluar con TTA en test final

---
## 🔧 1. Setup y GPU

In [ ]:
!pip install -q ultralytics>=8.4.0

import ultralytics
import torch
import os

print(f"✅ Ultralytics: {ultralytics.__version__}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA: {torch.version.cuda}")

# Info GPU
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"\n🖥️ GPU: {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")

# Recomendación automática
is_h100 = 'H100' in gpu_name

if vram_gb >= 70:
    if is_h100:
        rec_batch = 24
        rec_label = 'YOLO11l @ 1280 + batch=24 (H100) 🚀'
    else:
        rec_batch = 18
        rec_label = 'YOLO11l @ 1280 + batch=18 (A100 80GB) 🚀'
elif vram_gb >= 35:
    rec_batch = 14
    rec_label = 'YOLO11l @ 1280 + batch=14 (A100 40GB)'
elif vram_gb >= 20:
    rec_batch = 8
    rec_label = 'YOLO11l @ 1280 + batch=8 (L4)'
elif vram_gb >= 14:
    rec_batch = 4
    rec_label = 'YOLO11l @ 1280 + batch=4 (T4)'
else:
    rec_batch = 2
    rec_label = 'YOLO11l @ 1280 + batch=2 (GPU pequeña)'

print(f"\n💡 Recomendación: {rec_label}")

if is_h100:
    print(f"\n🚀 H100 detectada:")
    print(f"   • 2x más rápido vs A100")
    print(f"   • Batch óptimo: 18-24")

# Montar Drive
from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("\n✅ Drive montado")
else:
    print("\n✅ Drive ya montado")

---
## 📦 2. Extraer Dataset

In [ ]:
import zipfile
import glob
from tqdm import tqdm

DATASET_PATH = '/content/drive/MyDrive/SIRCCD_Dataset/sirccd_dataset_v1.0.0.zip'
EXTRACT_DIR = '/content/sirccd_dataset'

if os.path.exists(f'{EXTRACT_DIR}/data.yaml'):
    print("✅ Dataset ya extraído")
else:
    print("📦 Extrayendo dataset...")
    with zipfile.ZipFile(DATASET_PATH, 'r') as zip_ref:
        for file in tqdm(zip_ref.namelist(), desc="Extrayendo"):
            zip_ref.extract(file, '/content/')
    print("✅ Extraído")

# Conteo
print("\n📊 Dataset:")
total = 0
for split in ['train', 'val', 'test']:
    imgs = len(glob.glob(f'{EXTRACT_DIR}/images/{split}/*.jpg'))
    lbls = len(glob.glob(f'{EXTRACT_DIR}/labels/{split}/*.txt'))
    total += imgs
    print(f"   {split}: {imgs:,} imgs, {lbls:,} labels")
print(f"   TOTAL: {total:,} imágenes")

In [ ]:
# === DEDUPLICACIÓN - OMITIDA ===
#
# v1 tuvo MEJOR mAP sin deduplicación (mAP50=0.795 vs 0.78 con dedup)
# Imágenes "similares" actúan como augmentation natural.
# Variaciones sutiles ayudan a la generalización.

print("⏭️ Deduplicación omitida (mejor convergencia sin ella según v1)")

In [ ]:
import yaml

data_config = {
    'path': EXTRACT_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 2,
    'names': {0: 'bache', 1: 'grieta'}
}

with open(f'{EXTRACT_DIR}/data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ data.yaml listo (2 clases: bache, grieta)")

---
## 🧠 3. Seleccionar Modelo

Elige UNA opción descomentando la línea correspondiente.
Los pesos COCO se descargan automáticamente.

In [ ]:
from ultralytics import YOLO
from datetime import datetime

# ╔════════════════════════════════════════════════════════════╗
# ║  ELIGE TU MODELO (descomentar UNO)                         ║
# ╠════════════════════════════════════════════════════════════╣
# ║  YOLO11 - Recomendado para v4                              ║
# ╚════════════════════════════════════════════════════════════╝

MODEL_NAME = 'yolo11l.pt'     # 25.3M params |  87 GFLOPs | ~10 GB VRAM  ⭐ RECOMENDADO
# MODEL_NAME = 'yolo11x.pt'   # 56.9M params | 195 GFLOPs | ~16 GB VRAM

# ╔════════════════════════════════════════════════════════════╗
# ║  YOLO26 - Alternativa más nueva (NMS-free)                 ║
# ╚════════════════════════════════════════════════════════════╝

# MODEL_NAME = 'yolo26l.pt'   # 24.8M params |  86 GFLOPs | ~10 GB VRAM
# MODEL_NAME = 'yolo26x.pt'   # 55.7M params | 194 GFLOPs | ~16 GB VRAM

# ═════════════════════════════════════════════════════════════

# Cargar modelo con pesos COCO pre-entrenados
model = YOLO(MODEL_NAME)

model_tag = MODEL_NAME.replace('.pt', '')
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
experiment_name = f'v4-pretrained-{model_tag}_{timestamp}'

is_yolo26 = 'yolo26' in MODEL_NAME
if is_yolo26:
    print("🚀 YOLO26 detectado → Inferencia end-to-end (NMS-free)")

print(f"\n🧠 Modelo: {MODEL_NAME}")
print(f"📝 Experimento: {experiment_name}")
print(f"🖥️ GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.0f} GB)")

info = model.info()
print(f"\n📊 Arquitectura:")
print(f"   Parámetros: {info[1]/1e6:.1f}M")
print(f"   GFLOPs:     {info[2]:.1f}")
print(f"   Capas:      {info[0]}")

---
## ⚙️ 4. Auto-Batch (Encontrar batch óptimo)

In [ ]:
from ultralytics.utils.autobatch import check_train_batch_size
import psutil

# Resolución: 1280 para máxima precisión en grietas finas
IMGSZ = 1280

print(f"⚙️ Calculando batch size óptimo para imgsz={IMGSZ}...")
print("   (esto toma ~30 segundos)\n")

optimal_batch = check_train_batch_size(
    model=model.model,
    imgsz=IMGSZ,
    amp=True
)

ram_total_gb = psutil.virtual_memory().total / (1024**3)
ram_free_gb = psutil.virtual_memory().available / (1024**3)

is_h100 = 'H100' in gpu_name

if is_h100 and ram_free_gb > 200:
    safe_batch = 18
    CACHE_STRATEGY = 'ram'
    print(f"\n🚀 H100 + RAM abundante: batch={safe_batch}, cache=ram")
elif is_h100:
    safe_batch = max(2, int(optimal_batch * 0.75))
    safe_batch = (safe_batch // 2) * 2
    CACHE_STRATEGY = False
elif ram_free_gb > 200:
    safe_batch = max(2, int(optimal_batch * 0.75))
    safe_batch = (safe_batch // 2) * 2
    CACHE_STRATEGY = 'ram'
    print(f"\n💾 RAM abundante ({ram_free_gb:.0f} GB): cache=ram")
else:
    safe_batch = max(2, int(optimal_batch * 0.75))
    safe_batch = (safe_batch // 2) * 2
    CACHE_STRATEGY = 'disk'

print(f"\n📊 Resultados (imgsz={IMGSZ}):")
print(f"   Batch máximo detectado: {optimal_batch}")
print(f"   Batch final (seguro):    {safe_batch}")
print(f"   Cache strategy:          {CACHE_STRATEGY}")

print(f"\n📋 Referencia por GPU (YOLO11l @ 1280):")
print(f"   T4  (15GB): batch 2-4")
print(f"   L4  (24GB): batch 4-8")
print(f"   A100(40GB): batch 8-12")
print(f"   A100(80GB): batch 12-18 ⭐")
print(f"   H100(80GB): batch 18-24 🚀")

if safe_batch < 4:
    print(f"\n⚠️ Batch bajo ({safe_batch}). Gradient accumulation activado.")

BATCH_SIZE = safe_batch
print(f"\n✅ Batch final: {BATCH_SIZE}")

---
## 🚀 5. Hiperparámetros y Entrenamiento

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║             HIPERPARÁMETROS - AJUSTAR AQUÍ                  ║
# ╚════════════════════════════════════════════════════════════╝

EPOCHS = 100          # 100 para pruebas, 250 para producción
CONF_THRESHOLD = 0.001   # Validación (mAP completo)
CONF_INFERENCE = 0.30    # Producción (balance precision/recall)
PATIENCE = int(EPOCHS * 0.30)  # Early stopping (30% del total)

# Gradient accumulation para batch pequeño
if BATCH_SIZE <= 4:
    ACCUMULATE = max(1, 16 // BATCH_SIZE)
    print(f"📊 Gradient Accumulation: {ACCUMULATE}x")
    print(f"   Batch real: {BATCH_SIZE} → Effective: {BATCH_SIZE * ACCUMULATE}")
else:
    ACCUMULATE = 1

print(f"\n{'='*70}")
print(f"🚀 CONFIGURACIÓN v4 - YOLO11l Pre-entrenado COCO")
print(f"{'='*70}")
print(f"\n📋 Hiperparámetros:")
print(f"   Modelo:         {MODEL_NAME}")
print(f"   Epochs:         {EPOCHS} ({'PRUEBA' if EPOCHS < 200 else 'COMPLETO'})")
if EPOCHS < 200:
    print(f"                   (Aumentar a 250 para entrenamiento final)")
print(f"   Batch:          {BATCH_SIZE} (effective: {BATCH_SIZE * ACCUMULATE})")
print(f"   Resolución:     {IMGSZ}x{IMGSZ}")
print(f"   Optimizer:      auto")
print(f"   LR:             0.01 → 0.0015 (LRF=0.15, cosine)")
print(f"   Cache:          {CACHE_STRATEGY}")
print(f"\n🎯 Thresholds:")
print(f"   Conf (val):     {CONF_THRESHOLD}")
print(f"   Conf (prod):    {CONF_INFERENCE}")
print(f"   IOU (train):    0.7")
print(f"   Patience:       {PATIENCE} epochs")
print(f"{'='*70}")

### 🎮 Control: Pause y Resume

Puedes pausar el entrenamiento en cualquier momento:
- **Detener Colab**: Runtime → Interrupt execution
- **Reanudar**: Cambia `RESUME_TRAINING = True` y re-ejecuta

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║  CONTROL DE PAUSE/RESUME                                   ║
# ╚════════════════════════════════════════════════════════════╝

RESUME_TRAINING = False  # Cambiar a True para reanudar

checkpoint_path = None
if RESUME_TRAINING:
    import glob
    checkpoints = sorted(glob.glob(f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}/weights/last.pt'))
    if not checkpoints:
        checkpoints = sorted(glob.glob('/content/drive/MyDrive/SIRCCD_Models/v4-pretrained-*/weights/last.pt'))
    
    if checkpoints:
        checkpoint_path = checkpoints[-1]
        experiment_name = checkpoint_path.split('/')[-3]
        print(f"✅ MODO RESUME ACTIVADO")
        print(f"📍 Checkpoint: {checkpoint_path}")
        print(f"📝 Experimento: {experiment_name}")
        model = YOLO(checkpoint_path)
    else:
        print(f"❌ No se encontraron checkpoints. Iniciando desde cero.")
        RESUME_TRAINING = False
else:
    print(f"▶️  MODO NUEVO ENTRENAMIENTO (pesos COCO pre-entrenados)")
    print(f"\n💡 Para reanudar: RESUME_TRAINING = True")

In [ ]:
# ╔════════════════════════════════════════════════════════════╗
# ║             ENTRENAMIENTO v4                                ║
# ╚════════════════════════════════════════════════════════════╝

print(f"\n{'='*70}")
if RESUME_TRAINING:
    print(f"🔄 REANUDANDO ENTRENAMIENTO")
else:
    print(f"🚀 INICIANDO ENTRENAMIENTO v4 - {MODEL_NAME} (COCO pre-entrenado)")
print(f"{'='*70}\n")

results = model.train(
    # === Dataset ===
    data=f'{EXTRACT_DIR}/data.yaml',
    imgsz=IMGSZ,
    
    # === Training ===
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    
    # === Resume ===
    resume=RESUME_TRAINING,
    
    # === Optimizer ===
    optimizer='auto',
    lr0=0.01,
    lrf=0.15,                     # Fix plateau: LR final = 0.01 * 0.15 = 0.0015
    momentum=0.937,
    cos_lr=True,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    weight_decay=0.0005,
    
    # === Augmentation (optimizada para 1280) ===
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=3.0,
    perspective=0.0003,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.10,                   # Aumentado para más diversidad
    copy_paste=0.15,              # Aumentado para más variaciones
    erasing=0.3,
    close_mosaic=15,              # Últimos 15 epochs sin mosaic para refinar
    dropout=0.0,
    
    # === Thresholds (optimizado) ===
    iou=0.7,                      # Stricter bbox matching → mejor mAP50-95
    
    # === Guardado ===
    project='/content/drive/MyDrive/SIRCCD_Models',
    name=experiment_name,
    save_period=10,               # Checkpoint cada 10 epochs
    patience=PATIENCE,
    
    # === Validación ===
    val=True,
    val_period=5,
    plots=True,
    
    # === GPU/Performance ===
    device=0,
    amp=True,
    cache=CACHE_STRATEGY,
    workers=4,
    deterministic=True,
    seed=42,
    
    # === Transfer Learning ===
    pretrained=True,
    verbose=True,
    rect=False,
)

print(f"\n{'='*70}")
print(f"✅ ENTRENAMIENTO COMPLETADO - {MODEL_NAME} @ {IMGSZ}")
print(f"{'='*70}")
print(f"\n📁 Resultados en: SIRCCD_Models/{experiment_name}/")
print(f"   weights/best.pt  (mejor mAP50)")
print(f"   weights/last.pt  (para continuar)")

---
## 🔍 6. Evaluación

In [ ]:
import glob
import os
from ultralytics import YOLO

# Cargar best.pt
best_path = f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}/weights/best.pt'

if not os.path.exists(best_path):
    # Autodetectar
    candidates = sorted(glob.glob('/content/drive/MyDrive/SIRCCD_Models/v4-pretrained-*/weights/best.pt'))
    if candidates:
        best_path = candidates[-1]
        experiment_name = best_path.split('/')[-3]
        print(f"🔍 Autodetectado: {experiment_name}")
    else:
        raise FileNotFoundError("No se encontró best.pt")

best_model = YOLO(best_path)
print(f"✅ Modelo cargado: {best_path}")
print(f"   Tamaño: {os.path.getsize(best_path) / (1024*1024):.1f} MB")

In [ ]:
# Evaluación en validación
print("📊 Evaluando en validación...")
val_metrics = best_model.val(
    data=f'{EXTRACT_DIR}/data.yaml',
    split='val',
    conf=CONF_THRESHOLD,
    iou=0.6
)

print(f"\n🎯 Validación:")
print(f"   mAP50:     {val_metrics.box.map50:.4f}")
print(f"   mAP50-95:  {val_metrics.box.map:.4f}")
print(f"   Precision: {val_metrics.box.mp:.4f}")
print(f"   Recall:    {val_metrics.box.mr:.4f}")

In [ ]:
# Evaluación en test
print("📊 Evaluando en test...")
test_metrics = best_model.val(
    data=f'{EXTRACT_DIR}/data.yaml',
    split='test',
    conf=CONF_THRESHOLD,
    iou=0.6
)

print(f"\n🎯 Test:")
print(f"   mAP50:     {test_metrics.box.map50:.4f}")
print(f"   mAP50-95:  {test_metrics.box.map:.4f}")
print(f"   Precision: {test_metrics.box.mp:.4f}")
print(f"   Recall:    {test_metrics.box.mr:.4f}")

In [ ]:
# === COMPARACIÓN CON v1 ===
print("\n" + "="*70)
print("📈 COMPARACIÓN: v1 (YOLOv8m) vs v4 (YOLO11l pre-entrenado)")
print("="*70)

v1 = {'mAP50': 0.73388, 'mAP50-95': 0.45006, 'Precision': 0.73254, 'Recall': 0.68304}
v4_val = {
    'mAP50': val_metrics.box.map50, 'mAP50-95': val_metrics.box.map,
    'Precision': val_metrics.box.mp, 'Recall': val_metrics.box.mr
}
v4_test = {
    'mAP50': test_metrics.box.map50, 'mAP50-95': test_metrics.box.map,
    'Precision': test_metrics.box.mp, 'Recall': test_metrics.box.mr
}

print(f"\n{'Métrica':<12} {'v1 (YOLOv8m)':>14} {'v4 (val)':>10} {'v4 (test)':>10} {'Δ val':>8} {'Δ%':>8}")
print(f"{'-'*66}")
for key in v1:
    delta = v4_val[key] - v1[key]
    delta_pct = (delta / v1[key]) * 100
    sign = '+' if delta >= 0 else ''
    print(f"{key:<12} {v1[key]:>14.4f} {v4_val[key]:>10.4f} {v4_test[key]:>10.4f} "
          f"{sign}{delta:>7.4f} {sign}{delta_pct:>6.1f}%")

# Por clase
print(f"\n📊 Métricas por Clase (test):")
print(f"{'Clase':<10} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print(f"{'-'*50}")
for i, name in enumerate(['bache', 'grieta']):
    print(f"{name:<10} {test_metrics.box.p[i]:>10.3f} {test_metrics.box.r[i]:>10.3f} "
          f"{test_metrics.box.ap50[i]:>10.3f} {test_metrics.box.ap[i]:>10.3f}")

In [ ]:
# === TEST-TIME AUGMENTATION (TTA) ===
print("\n🔬 Evaluación con TTA (Test-Time Augmentation):")
tta_metrics = best_model.val(
    data=f'{EXTRACT_DIR}/data.yaml',
    split='test',
    augment=True,
    conf=CONF_THRESHOLD,
    iou=0.6
)

print(f"\n📊 Comparación (test):")
print(f"   {'':>12} {'Sin TTA':>10} {'Con TTA':>10} {'Δ':>8} {'Δ%':>8}")
print(f"   {'-'*48}")
for label, no_tta, tta in [
    ('mAP50', test_metrics.box.map50, tta_metrics.box.map50),
    ('mAP50-95', test_metrics.box.map, tta_metrics.box.map),
    ('Precision', test_metrics.box.mp, tta_metrics.box.mp),
    ('Recall', test_metrics.box.mr, tta_metrics.box.mr),
]:
    d = tta - no_tta
    d_pct = (d / no_tta) * 100 if no_tta > 0 else 0
    s = '+' if d >= 0 else ''
    print(f"   {label:<12} {no_tta:>10.4f} {tta:>10.4f} {s}{d:>7.4f} {s}{d_pct:>6.2f}%")

---
## 📈 7. Visualizaciones

In [ ]:
from IPython.display import Image, display

results_dir = f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}'

for filename, title in [
    ('results.png', '📈 Curvas de Entrenamiento'),
    ('confusion_matrix_normalized.png', '🔢 Matriz de Confusión'),
    ('F1_curve.png', '📊 Curva F1'),
    ('PR_curve.png', '📊 Precision-Recall'),
]:
    path = os.path.join(results_dir, filename)
    if os.path.exists(path):
        print(f"\n{title}:")
        display(Image(filename=path, width=800))

In [ ]:
import random

# Predicciones en test aleatorio
test_images = glob.glob(f'{EXTRACT_DIR}/images/test/*.jpg')
sample = random.sample(test_images, min(12, len(test_images)))

print(f"🔍 Predicciones de ejemplo (conf > {CONF_INFERENCE}):")

results = best_model.predict(
    source=sample,
    conf=CONF_INFERENCE,
    iou=0.5,
    save=True,
    project='/content/predictions_v4',
    name='test',
    exist_ok=True
)

total_dets = 0
for r in results:
    fname = os.path.basename(r.path)
    num = len(r.boxes)
    total_dets += num
    if num > 0:
        det = ', '.join(f"{r.names[int(c)]}({float(conf):.2f})"
                       for c, conf in zip(r.boxes.cls, r.boxes.conf))
        print(f"  ✅ {fname}: {det}")
    else:
        print(f"  ⬜ {fname}: Sin detecciones")

print(f"\n📊 Resumen: {total_dets} detecciones en {len(sample)} imágenes")
print(f"   Promedio: {total_dets/len(sample):.1f} detecciones/imagen")

# Mostrar algunas
pred_imgs = sorted(glob.glob('/content/predictions_v4/test/*.jpg'))[:6]
for p in pred_imgs:
    display(Image(filename=p, width=500))

---
## 💾 8. Exportar para Producción

In [ ]:
import shutil

export_dir = f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}/exports'
os.makedirs(export_dir, exist_ok=True)

# PyTorch
shutil.copy(best_path, f'{export_dir}/best.pt')
print(f"✅ PyTorch: {export_dir}/best.pt")

# ONNX
onnx_path = best_model.export(format='onnx', imgsz=IMGSZ, simplify=True, opset=17)
shutil.copy(onnx_path, f'{export_dir}/best.onnx')
print(f"✅ ONNX: {export_dir}/best.onnx")

# TorchScript
ts_path = best_model.export(format='torchscript', imgsz=IMGSZ)
shutil.copy(ts_path, f'{export_dir}/best.torchscript')
print(f"✅ TorchScript: {export_dir}/best.torchscript")

# Tamaños
print(f"\n📊 Tamaños:")
for f in os.listdir(export_dir):
    size = os.path.getsize(os.path.join(export_dir, f)) / (1024*1024)
    print(f"   {f}: {size:.1f} MB")

---
## 📝 9. Resumen Final

In [ ]:
import json
from datetime import datetime

summary = {
    'version': 'v4-pretrained',
    'fecha': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'estrategia': 'Transfer learning YOLO11l pre-entrenado COCO @ 1280',
    'modelo': MODEL_NAME,
    'parametros': f'{info[1]/1e6:.1f}M',
    'gflops': f'{info[2]:.1f}',
    'clases': ['bache', 'grieta'],
    'configuracion': {
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'imgsz': IMGSZ,
        'optimizer': 'auto',
        'lr0': 0.01,
        'lrf': 0.15,
        'cos_lr': True,
        'patience': PATIENCE,
        'val_period': 5,
        'conf_threshold': CONF_THRESHOLD,
        'conf_inference': CONF_INFERENCE,
        'iou_threshold': 0.7
    },
    'gpu': torch.cuda.get_device_name(0),
    'deduplicacion_omitida': True,
    'metricas': {
        'v1_baseline': {'mAP50': 0.73388, 'mAP50-95': 0.45006, 'P': 0.73254, 'R': 0.68304},
        'v4_val': {
            'mAP50': float(val_metrics.box.map50),
            'mAP50-95': float(val_metrics.box.map),
            'P': float(val_metrics.box.mp),
            'R': float(val_metrics.box.mr)
        },
        'v4_test': {
            'mAP50': float(test_metrics.box.map50),
            'mAP50-95': float(test_metrics.box.map),
            'P': float(test_metrics.box.mp),
            'R': float(test_metrics.box.mr)
        },
        'v4_test_tta': {
            'mAP50': float(tta_metrics.box.map50),
            'mAP50-95': float(tta_metrics.box.map),
            'P': float(tta_metrics.box.mp),
            'R': float(tta_metrics.box.mr)
        }
    }
}

summary_path = f'/content/drive/MyDrive/SIRCCD_Models/{experiment_name}/training_summary_v4.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# === RESUMEN VISUAL ===
print("\n" + "="*70)
print("🎉 ENTRENAMIENTO v4 - RESUMEN FINAL")
print("="*70)
print(f"\n🧠 Modelo:  {MODEL_NAME}")
print(f"📊 Params:  {info[1]/1e6:.1f}M")
print(f"🔄 Epochs:  {EPOCHS}")
print(f"📐 ImgSz:   {IMGSZ}")
print(f"🖥️ GPU:     {torch.cuda.get_device_name(0)}")
print(f"🎯 Clases:  bache, grieta")

print(f"\n{'Métrica':<12} {'v1':>10} {'v4 val':>10} {'v4 test':>10} {'v4+TTA':>10} {'Δ%':>8}")
print(f"{'-'*63}")
for key, k1, k4 in [
    ('mAP50', 'mAP50', 'mAP50'),
    ('mAP50-95', 'mAP50-95', 'mAP50-95'),
    ('Precision', 'P', 'P'),
    ('Recall', 'R', 'R')
]:
    v1_val = summary['metricas']['v1_baseline'][k1]
    v4_test_val = summary['metricas']['v4_test'][k4]
    delta_pct = ((v4_test_val - v1_val) / v1_val) * 100
    sign = '+' if delta_pct >= 0 else ''
    
    print(f"{key:<12} {v1_val:>10.4f} "
          f"{summary['metricas']['v4_val'][k4]:>10.4f} "
          f"{v4_test_val:>10.4f} "
          f"{summary['metricas']['v4_test_tta'][k4]:>10.4f} "
          f"{sign}{delta_pct:>6.1f}%")

print(f"\n💾 Guardado en: SIRCCD_Models/{experiment_name}/")
print(f"📄 Resumen JSON: training_summary_v4.json")
print("="*70)

---
## 🔮 10. ¿Qué Sigue?

### ⚡ Si usaste EPOCHS = 100 (prueba):

1. **Aumentar a 250 epochs**:
   - Cambiar `EPOCHS = 250`
   - Cambiar `RESUME_TRAINING = True`
   - Re-ejecutar desde la celda de control hasta el final

2. **O continuar desde checkpoint**:
   ```python
   EPOCHS = 250
   RESUME_TRAINING = True
   ```

### 📊 Si v4 superó a v1:

1. Descargar `best.pt` desde Drive
2. Colocar en `ml/models/v4/best.pt`
3. Actualizar `backend/.env`:
   ```
   YOLO_MODEL_PATH=../ml/models/v4/best.pt
   ```
4. Actualizar `MLInferenceService` con las nuevas métricas
5. Reiniciar backend

### 🎯 Para aún más precisión:

- **Más epochs**: 250 → 300
- **YOLO11x**: Más parámetros (56.9M vs 25.3M)
- **YOLO26l**: Arquitectura más nueva con NMS-free
- **Ensemble**: Combinar v1 + v4 con Weighted Boxes Fusion
- **Más datos**: CRACK500 + CFD + SUT-Crack